# snowdrift: daily schema drift check (Snowflake)

Compares the source system (Teradata) with its copies in Snowflake and Databricks, flags what breaks, drafts fix SQL for review, and keeps history in Snowflake tables.

**Needs:** a Snowflake Notebook in Workspaces running **Python 3.12**. Older (legacy) Snowflake notebooks run Python 3.9/3.10 and can't load snowdrift.

Snowflake notebooks can only see Snowflake, so Teradata and Databricks metadata come in as CSV exports on a stage. Fill in every `<...>` below.

In [ ]:
# Install. With an external access integration for GitHub:
#   !pip install git+https://github.com/jennigirl62/snowdrift.git
# Without internet access: run `python tools/build_zip.py`, upload dist/snowdrift.zip to an INTERNAL stage
# (Snowsight upload, or PUT with AUTO_COMPRESS=FALSE), then:
import gzip
import os
import shutil
import sys

from snowflake.snowpark.context import get_active_session

session = get_active_session()
CODE_STAGE = "@<DB>.<SCHEMA>.<STAGE>"
session.file.get(f"{CODE_STAGE}/snowdrift.zip", "/tmp/snowdrift")
if os.path.exists("/tmp/snowdrift/snowdrift.zip.gz"):  # uploaded with PUT's default compression
    with gzip.open("/tmp/snowdrift/snowdrift.zip.gz") as src, open("/tmp/snowdrift/snowdrift.zip", "wb") as dst:
        shutil.copyfileobj(src, dst)
sys.path.insert(0, "/tmp/snowdrift/snowdrift.zip")

In [ ]:
from snowdrift.ai import CortexClient
from snowdrift.notebook import check
from snowdrift.sources import CsvSource, SnowflakeSource, TeradataExportSource, snowflake_views, snowpark_runner, stage_reader
from snowdrift.store import SnowparkBackend

# ---- What to compare ---------------------------------------------------------
TABLES = ["CUSTOMERS", "JOBS", "INVOICES"]  # start with 10-20 tables
DATABASE = "<DB>"  # database holding the Snowflake copies
SF_SCHEMAS = ["<FINAL_SCHEMA>", "<WORK_SCHEMA>"]
SF_WORK_SCHEMAS = ["<WORK_SCHEMA>"]
HISTORY_SCHEMA = "<DB>.SNOWDRIFT"  # where snowdrift keeps its own history
EXPORT_STAGE = "@<DB>.<SCHEMA>.<STAGE>"  # internal stage holding teradata_columns.csv and databricks_columns.csv
CORTEX_MODEL = "<cortex-model-name>"  # or None for template explanations

session.sql(f"CREATE SCHEMA IF NOT EXISTS {HISTORY_SCHEMA}").collect()

In [ ]:
read = stage_reader(session)
sources = {
    "teradata": TeradataExportSource("teradata", f"{EXPORT_STAGE}/teradata_columns.csv", read_text=read),
    "snowflake": SnowflakeSource(snowpark_runner(session), DATABASE, SF_SCHEMAS, work_schemas=SF_WORK_SCHEMAS),
    # Databricks: an export of sql/databricks_columns.sql, uploaded to the same stage (optional)
    "databricks": CsvSource(
        "databricks", f"{EXPORT_STAGE}/databricks_columns.csv", work_schemas=["<work_schema>"], read_text=read
    ),
}

In [ ]:
report = check(
    sources=sources,
    reference="teradata",
    targets=["snowflake", "databricks"],
    tables=TABLES,
    store=SnowparkBackend(session, HISTORY_SCHEMA),
    ignore_columns=["<LOAD_AUDIT_COLUMN>"],  # columns that only exist in the copies
    key_columns={"JOBS": ["JOB_ID"]},  # merge/upsert keys per table
    expected_collation={"snowflake": "en-ci"},  # remove if you don't set collations
    views=snowflake_views(snowpark_runner(session), DATABASE, SF_SCHEMAS),
    llm=CortexClient(session, CORTEX_MODEL) if CORTEX_MODEL else None,
)

## Results as a table (sort, filter, export)

In [ ]:
import pandas as pd

pd.DataFrame(report.to_rows()) if report.events else "No drift at or above the reporting threshold."

## Mark a false positive
Copy a `drift_id` from the table above. It stays quiet until that column changes again.

In [ ]:
# from snowdrift.store import Store
# Store(SnowparkBackend(session, HISTORY_SCHEMA)).set_status("<drift_id>", "false_positive")